# ABook Audiobook Studio

Free Kaggle production runner for *Hear All, See All, Say Nothing*. Start with the short narrator approval sample. Only switch to a full run after the manuscript, voice and pronunciations are approved. Generated audio is saved as notebook output and as a resumable checkpoint ZIP.

In [ ]:
# Change only these settings.
RUN_MODE = "sample"       # "sample" or "full"
VOICE_MODE = "narrator"  # "narrator" or "pov"
REPO_URL = "https://github.com/simplebusiness26/ABook.git"
BRANCH = "main"

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/ABook")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def run(command):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(item) for item in command], check=True)

if (REPO_DIR / ".git").exists():
    run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
else:
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-qq", "-y", "espeak-ng", "ffmpeg"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", REPO_DIR / "audiobook/requirements-kaggle.txt"]])

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle Notebook options, enable a GPU accelerator, then restart and Run all.")
print("GPU ready:", torch.cuda.get_device_name(0))

In [ ]:
pipeline = REPO_DIR / "audiobook/scripts/run_pipeline.py"
run([
    sys.executable, pipeline,
    "--mode", RUN_MODE,
    "--voice-mode", VOICE_MODE,
    "--device", "cuda",
    "--auto-restore",
])

In [ ]:
from IPython.display import Audio, FileLink, display

exports = REPO_DIR / "audiobook/exports"
samples = sorted((exports / "approval-samples").glob("*.mp3"))
if samples:
    print("Listen to the approval sample:")
    display(Audio(filename=str(samples[-1])))
    display(FileLink(str(samples[-1])))

m4b_files = sorted(exports.glob("*.m4b"))
for path in m4b_files:
    display(FileLink(str(path)))

checkpoint = exports / "abook-audiobook-checkpoint.zip"
if checkpoint.exists():
    print("Keep this file if the full run needs another Kaggle session:")
    display(FileLink(str(checkpoint)))

print("Use Save Version so Kaggle retains these outputs.")